In [1]:
import random
import json
import math
import os

class CoCInvestigator:
    def __init__(self):
        self.name = "Unknown"
        self.age = 20
        self.occupation = "Unknown"
        self.stats = {
            "STR": 0, "CON": 0, "SIZ": 0, "DEX": 0,
            "APP": 0, "INT": 0, "POW": 0, "EDU": 0, "LUCK": 0
        }
        self.derived_stats = {
            "HP": 0, "MP": 0, "SAN": 0, "MOV": 8, "DB": "0", "BUILD": 0
        }
        self.background = ""

    def roll_dice(self, num, sides):
        """模拟投掷 num 个 sides 面骰子"""
        return sum(random.randint(1, sides) for _ in range(num))

    def calculate_derived_stats(self):
        """计算第七版规则下的衍生属性"""
        self.derived_stats["HP"] = math.floor((self.stats["CON"] + self.stats["SIZ"]) / 10)
        self.derived_stats["MP"] = math.floor(self.stats["POW"] / 5)
        self.derived_stats["SAN"] = self.stats["POW"]

        # 计算体格(Build)和伤害加值(Damage Bonus, DB)
        str_siz = self.stats["STR"] + self.stats["SIZ"]
        if 2 <= str_siz <= 64:
            self.derived_stats["DB"] = "-2"
            self.derived_stats["BUILD"] = -2
        elif 65 <= str_siz <= 84:
            self.derived_stats["DB"] = "-1"
            self.derived_stats["BUILD"] = -1
        elif 85 <= str_siz <= 124:
            self.derived_stats["DB"] = "0"
            self.derived_stats["BUILD"] = 0
        elif 125 <= str_siz <= 164:
            self.derived_stats["DB"] = "+1D4"
            self.derived_stats["BUILD"] = 1
        elif 165 <= str_siz <= 204:
            self.derived_stats["DB"] = "+1D6"
            self.derived_stats["BUILD"] = 2
        else:
            self.derived_stats["DB"] = "+2D6" # 简化的超大体格处理
            self.derived_stats["BUILD"] = 3

        # 计算移动力(MOV)
        if self.stats["DEX"] < self.stats["SIZ"] and self.stats["STR"] < self.stats["SIZ"]:
            self.derived_stats["MOV"] = 7
        elif self.stats["DEX"] > self.stats["SIZ"] and self.stats["STR"] > self.stats["SIZ"]:
            self.derived_stats["MOV"] = 9
        else:
            self.derived_stats["MOV"] = 8

        # 年龄对MOV的影响（简化版：40岁以上逐渐减1，此处省略详细年龄惩罚计算以保简洁）

    def generate_random(self):
        """系统随机生成属性 (COC 7th 标准规则)"""
        print("\n--- 正在为您掷骰生成属性 ---")
        self.stats["STR"] = self.roll_dice(3, 6) * 5
        self.stats["CON"] = self.roll_dice(3, 6) * 5
        self.stats["DEX"] = self.roll_dice(3, 6) * 5
        self.stats["APP"] = self.roll_dice(3, 6) * 5
        self.stats["POW"] = self.roll_dice(3, 6) * 5

        self.stats["SIZ"] = (self.roll_dice(2, 6) + 6) * 5
        self.stats["INT"] = (self.roll_dice(2, 6) + 6) * 5
        self.stats["EDU"] = (self.roll_dice(2, 6) + 6) * 5

        self.stats["LUCK"] = self.roll_dice(3, 6) * 5

        self.calculate_derived_stats()
        self.display_character()

    def generate_manual(self):
        """用户完全指定属性（购点或面团带入）"""
        print("\n--- 请手动输入各项属性 (推荐范围 15-90) ---")
        for attr in self.stats.keys():
            while True:
                try:
                    val = int(input(f"请输入 {attr} 的值: "))
                    if 0 <= val <= 100:
                        self.stats[attr] = val
                        break
                    else:
                        print("属性值应在 0 到 100 之间，请重新输入。")
                except ValueError:
                    print("无效输入，请输入整数。")
        self.calculate_derived_stats()
        self.display_character()

    def modify_attribute(self, attr_name, new_value):
        """修改单一属性并级联更新"""
        attr_name = attr_name.upper()
        if attr_name in self.stats:
            self.stats[attr_name] = new_value
            self.calculate_derived_stats()
            print(f"属性 {attr_name} 已更新为 {new_value}。")
        elif attr_name in self.derived_stats:
            self.derived_stats[attr_name] = new_value
            print(f"衍生属性 {attr_name} 已强制更新为 {new_value}。")
        else:
            print(f"找不到属性：{attr_name}")

    def export_to_json(self, filename="character_sheet.json"):
        """将角色数据导出为 JSON 文件"""
        data = {
            "Personal_Info": {
                "Name": self.name,
                "Age": self.age,
                "Occupation": self.occupation
            },
            "Stats": self.stats,
            "Derived_Stats": self.derived_stats,
            "Background": self.background
        }
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
        print(f"\n[成功] 角色卡已导出至 {filename}")

    def generate_background_with_llm(self):
        """
        [伪代码] 使用 LLM 接口辅助创作角色背景
        适合结合 LangChain 或自定义 Agent 框架进行 RAG/Prompt 注入
        """
        print("\n--- 正在请求 LLM 接口生成角色背景... ---")

        # 1. 组装 Prompt
        prompt = f"""
        你是一个克苏鲁的呼唤(Call of Cthulhu)的守秘人(KP)。
        请根据以下调查员的属性，为其撰写一段约200字的洛夫克拉夫特风格的角色背景故事。
        姓名: {self.name}
        职业: {self.occupation}
        核心属性: {json.dumps(self.stats)}
        注意：如果角色的INT(智力)很高但SAN(理智)较低，请体现出其对神秘学或禁忌知识的执迷；如果STR(力量)很高，可以强调其体格特征。
        """

        # 2. 调用大模型 API (此处为伪代码替代)
        # response = llm_client.chat.completions.create(
        #     model="gpt-4", # 或其他模型名称
        #     messages=[{"role": "user", "content": prompt}]
        # )
        # generated_text = response.choices[0].message.content

        # 3. 模拟返回结果
        generated_text = (
            f"作为一名{self.occupation}，{self.name} 始终觉得这个世界表面下隐藏着某种不和谐的律动。 "
            f"TA 那高达 {self.stats['EDU']} 的教育水平让TA接触到了常人无法触及的古老卷宗。 "
            f"最近，伴随着长期的失眠，{self.name} 决定追寻那来自深渊的呼唤..."
        )

        self.background = generated_text
        print(f"\n[生成的背景故事]:\n{self.background}\n")

    def display_character(self):
        """打印当前角色卡信息"""
        print(f"\n{'='*30}")
        print(f"调查员: {self.name} | 职业: {self.occupation} | 年龄: {self.age}")
        print(f"{'-'*30}")
        print("【核心属性】")
        for k, v in self.stats.items():
            print(f"{k}: {v}", end="\t")
            if list(self.stats.keys()).index(k) % 3 == 2: print() # 换行排版
        print(f"\n{'-'*30}")
        print("【衍生属性】")
        for k, v in self.derived_stats.items():
            print(f"{k}: {v}", end="\t")
        print(f"\n{'='*30}\n")

def interactive_creation():
    """交互式创建角色的流程主函数"""
    print("欢迎来到《克苏鲁的呼唤》7版 调查员创建系统")
    char = CoCInvestigator()

    char.name = input("请输入调查员姓名: ") or "John Doe"
    char.occupation = input("请输入职业 (如 医生, 私家侦探, 教授): ") or "私家侦探"
    try:
        char.age = int(input("请输入年龄: ") or "25")
    except ValueError:
        char.age = 25

    print("\n请选择属性决定方式：")
    print("1. 系统自动掷骰 (3D6*5 等标准规则)")
    print("2. 手动完全指定 (面团导入)")
    choice = input("请输入选项 (1 或 2): ")

    if choice == '2':
        char.generate_manual()
    else:
        char.generate_random()

    while True:
        print("\n接下来您希望做什么？")
        print("1. 修改特定属性 (例如修改 LUCK)")
        print("2. 使用 LLM 生成角色背景故事")
        print("3. 导出角色卡为 JSON")
        print("4. 查看当前角色卡")
        print("0. 结束车卡并退出")

        action = input("请选择操作: ")

        if action == '1':
            attr = input("请输入要修改的属性缩写 (如 STR, LUCK, HP): ")
            try:
                val = int(input(f"请输入新的值: "))
                char.modify_attribute(attr, val)
            except ValueError:
                print("无效的数值！")
        elif action == '2':
            char.generate_background_with_llm()
        elif action == '3':
            file_name = input("请输入保存的文件名 (默认 character.json): ") or "character.json"
            char.export_to_json(file_name)
        elif action == '4':
            char.display_character()
        elif action == '0':
            print("调查员准备完毕。愿你在黑暗中幸存...")
            break
        else:
            print("未知选项，请重试。")

if __name__ == "__main__":
    # 启动交互式生成
    interactive_creation()

欢迎来到《克苏鲁的呼唤》7版 调查员创建系统

请选择属性决定方式：
1. 系统自动掷骰 (3D6*5 等标准规则)
2. 手动完全指定 (面团导入)

--- 正在为您掷骰生成属性 ---

调查员: 亚楠 | 职业: 学生 | 年龄: 20
------------------------------
【核心属性】
STR: 65	CON: 25	SIZ: 45	
DEX: 60	APP: 60	INT: 70	
POW: 60	EDU: 80	LUCK: 30	

------------------------------
【衍生属性】
HP: 7	MP: 12	SAN: 60	MOV: 9	DB: 0	BUILD: 0	


接下来您希望做什么？
1. 修改特定属性 (例如修改 LUCK)
2. 使用 LLM 生成角色背景故事
3. 导出角色卡为 JSON
4. 查看当前角色卡
0. 结束车卡并退出

调查员: 亚楠 | 职业: 学生 | 年龄: 20
------------------------------
【核心属性】
STR: 65	CON: 25	SIZ: 45	
DEX: 60	APP: 60	INT: 70	
POW: 60	EDU: 80	LUCK: 30	

------------------------------
【衍生属性】
HP: 7	MP: 12	SAN: 60	MOV: 9	DB: 0	BUILD: 0	


接下来您希望做什么？
1. 修改特定属性 (例如修改 LUCK)
2. 使用 LLM 生成角色背景故事
3. 导出角色卡为 JSON
4. 查看当前角色卡
0. 结束车卡并退出

--- 正在请求 LLM 接口生成角色背景... ---

[生成的背景故事]:
作为一名学生，亚楠 始终觉得这个世界表面下隐藏着某种不和谐的律动。 TA 那高达 80 的教育水平让TA接触到了常人无法触及的古老卷宗。 最近，伴随着长期的失眠，亚楠 决定追寻那来自深渊的呼唤...


接下来您希望做什么？
1. 修改特定属性 (例如修改 LUCK)
2. 使用 LLM 生成角色背景故事
3. 导出角色卡为 JSON
4. 查看当前角色卡
0. 结束车卡并退出
调查员准备完毕。愿你在黑暗中幸存...
